# System prompts y parámetros en local

Hasta ahora habia usado system prompts con la API de Anthropic.
Aquí hacemos lo mismo con Ollama — mismo concepto, modelo local, sin coste por token.

La ventaja: podemos experimentar con temperatura y otros parámetros
libremente sin preocuparnos del gasto.

## Qué veremos
- Cómo el system prompt cambia el comportamiento del modelo
- Qué efecto tiene la temperatura en las respuestas
- Diferencia entre un modelo "sin personalidad" y uno bien configurado

In [1]:
import httpx

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "qwen2.5:7b"

def chat(user_msg: str, system: str = "", temperature: float = 0.7) -> str:
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": user_msg})

    response = httpx.post(
        OLLAMA_URL,
        json={
            "model": MODEL,
            "messages": messages,
            "stream": False,
            "options": {"temperature": temperature}
        },
        timeout=60
    )
    return response.json()["message"]["content"]

## El system prompt cambia todo

Sin system prompt el modelo responde con su personalidad por defecto.
Con system prompt le damos un rol, restricciones y contexto permanente.
Mismo input, respuesta completamente diferente.

In [2]:
pregunta = "¿Cómo soluciono un error 500 en mi API?"

# Sin system prompt
sin_sistema = chat(pregunta)
print("SIN SYSTEM PROMPT:")
print(sin_sistema)
print("\n" + "="*50 + "\n")

# Con system prompt
con_sistema = chat(
    pregunta,
    system="""Eres un senior backend engineer especializado en APIs REST.
Respondes de forma concisa y técnica, siempre en bullet points.
Nunca das explicaciones largas — solo pasos accionables."""
)
print("CON SYSTEM PROMPT:")
print(con_sistema)

SIN SYSTEM PROMPT:
Un código de error 500 (Internal Server Error) indica que hay un problema interno en el servidor que está causando que la solicitud no pueda completarse correctamente. Este error es generalmente una respuesta obvia del servidor, pero puede ser útil investigar más a fondo para identificar el problema específico.

Aquí te dejo algunos pasos que puedes seguir para diagnosticar y resolver este error:

1. **Verifica los registros del servidor**:
   - Mira la consola de errores o los logs del servidor donde se encuentra tu API.
   - Estos logs pueden proporcionarte más detalles sobre el problema subyacente.

2. **Prueba en diferentes ambientes**:
   - Si estás utilizando un entorno de desarrollo y producción, prueba la API en ambos para ver si el problema es específico a un ambiente u otro.

3. **Revisa los recursos del servidor**:
   - Asegúrate de que tu servidor tenga suficiente memoria RAM y CPU.
   - Comprueba también si hay problemas con la capacidad de almacenamient

## Temperatura — aleatoriedad controlada

Controla cuánta variedad introduce el modelo al generar texto.

- `temperature=0` → siempre la misma respuesta (determinista)
- `temperature=0.2` → casi determinista, ligera variedad
- `temperature=0.7` → creativo y variable, útil para brainstorming

Ejecutamos el mismo prompt tres veces con temperature=0
para demostrar el determinismo, y tres veces con temperature=0.7
para ver la variedad.

In [3]:
prompt = "Dame un nombre creativo para una startup de IA"
system = "Eres un experto en branding tecnológico."

print("=== TEMPERATURE 0 (determinista) ===")
for i in range(3):
    r = chat(prompt, system=system, temperature=0)
    print(f"[{i+1}] {r}\n")

print("=== TEMPERATURE 0.7 (creativo) ===")
for i in range(3):
    r = chat(prompt, system=system, temperature=0.7)
    print(f"[{i+1}] {r}\n")

=== TEMPERATURE 0 (determinista) ===
[1] Claro, aquí tienes algunas sugerencias creativas para tu startup de IA:

1. **NexaMind**
2. **SynthetixAI**
3. **CogniWave**
4. **NeuroNetra**
5. **VirtuMind**
6. **AetherIQ**
7. **PulseIntel**
8. **QuantumSage**
9. **Evolvix**
10. **NexaLumen**

Estos nombres combinan elementos que sugieren inteligencia artificial, tecnología avanzada y progreso futuro, lo cual puede ser atractivo para tu público objetivo.

[2] Claro, aquí tienes algunas sugerencias creativas para tu startup de IA:

1. **NexaMind**
2. **SynthetixAI**
3. **CogniWave**
4. **NeuroNetra**
5. **VirtuMind**
6. **AetherIQ**
7. **PulseIntel**
8. **QuantumSage**
9. **Evolvix**
10. **NexaLumen**

Estos nombres combinan elementos que sugieren inteligencia artificial, tecnología avanzada y progreso futuro, lo cual puede ser atractivo para tu público objetivo.

[3] Claro, aquí tienes algunas sugerencias creativas para tu startup de IA:

1. **NexaMind**
2. **SynthetixAI**
3. **CogniWave**
4.

## Conversación multi-turno — memoria entre mensajes

Hasta ahora cada llamada es independiente: el modelo no recuerda
lo que dijiste antes. Para mantener contexto hay que pasar
el historial completo de mensajes en cada llamada.

Es exactamente lo que hace el chat de Claude internamente —
no hay magia, solo acumulación del historial.

In [5]:
def chat_con_memoria(system: str = ""):
    historial = []
    if system:
        historial.append({"role": "system", "content": system})

    print("Chat iniciado (escribe 'salir' para terminar)\n")

    while True:
        user_input = input("Tú: ")
        if user_input.lower() == "salir":
            break

        historial.append({"role": "user", "content": user_input})

        response = httpx.post(
            OLLAMA_URL,
            json={"model": MODEL, "messages": historial, "stream": False},
            timeout=60
        )

        assistant_msg = response.json()["message"]["content"]
        historial.append({"role": "assistant", "content": assistant_msg})

        print(f"\nModelo: {assistant_msg}\n")

    return historial

historial = chat_con_memoria(system="Eres un asistente técnico conciso.")

Chat iniciado (escribe 'salir' para terminar)


Modelo: Para poder ayudarte a responder esa pregunta, necesito que me indiques tu nombre. ¿Cómo te llamas?


Modelo: Mucho gusto, Alex. ¿En qué puedo ayudarte hoy?


Modelo: Te llamas Alex. ¿Necesitas ayuda con algo más, Alex?

